In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.optim as optim
import numpy as np
import pandas as pd
import seaborn as sns
import datetime
import itertools
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import time
pd.options.mode.chained_assignment = None  # type: ignore # default='warn'
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import random

In [2]:
# !pip list

In [3]:
print(torch.__version__)

2.0.1+cu117


In [4]:
from DAST_Network import *
from DAST_preprocessing import *
from DAST_Train_general_batch import *

In [5]:
train_FD001 = pd.read_csv("CMAPSS/train_FD001_preprocessed.csv", index_col = 0)
# train_FD004 = pd.read_csv("CMAPSS/train_FD004_preprocessed.csv", index_col = 0)

In [6]:
train_FD001 = censored_target(train_FD001, 50)
unit_train, unit_valid = train_test_split(train_FD001.unit_number.unique(), test_size = 0.2)

In [7]:
# np.sort(unit_valid)

In [8]:
# np.sort(train_001.unit_number.unique())

In [9]:
valid_FD001 = train_FD001.loc[train_FD001.unit_number.isin(unit_valid)]
train_FD001 = train_FD001.loc[train_FD001.unit_number.isin(unit_train)]

In [10]:
quanti = ['operational_setting_1',
       'operational_setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
       'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13',
       'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
quali = ['unit_number', 'time_cycle', 'rul']
target = ['target_10', 'target_20', 'target_30', 'target_40', 'target_50',
       'target_60', 'target_70', 'target_80', 'target_90', 'target_100',
       'target_110', 'target_120', 'target_130']
window_Size = 40
mu, sigma = train_FD001[quanti].mean(), train_FD001[quanti].std()

In [11]:
new_train_FD001, target_train_FD001 = sliding_window(train_FD001, quali, quanti, target, window_Size)
new_valid_FD001, target_valid_FD001 = sliding_window_2(valid_FD001, quali, quanti, target, window_Size, mu, sigma)

In [12]:
new_train_FD001 = statistical_feature(new_train_FD001, window_Size, len(quanti))
new_valid_FD001 = statistical_feature(new_valid_FD001, window_Size, len(quanti))

In [13]:
print('train shape', new_train_FD001.shape)
print('valid shape', new_valid_FD001.shape)

valid shape (3008, 42, 19)
train shape (10068, 42, 19)


In [15]:
DAST = DAST_OR(dim_val_s = 64, dim_attn_s = 64, dim_val_t = 64, dim_attn_t = 64, dim_val = 64, dim_attn = 64,
               time_step = window_Size+2, input_size = len(quanti), dec_seq_len = 10, out_seq_len = 13, n_decoder_layers = 1,
               n_encoder_layers = 2, n_heads = 4, dropout = 0.2)
# Loss :
criterion = nn.BCELoss(reduction='sum')
# Optimizer :
optimizer = torch.optim.RAdam(DAST.parameters(), lr = 0.001)
# Num Epoch
num_epochs = 101
# Batch size
batch_size = 256
# Scheduler :
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001, steps_per_epoch=len(new_train_FD001), epochs=num_epochs)
# Training models :
train_DAST(new_train_FD001, target_train_FD001, new_valid_FD001, target_valid_FD001, DAST, criterion, optimizer, scheduler, num_epochs, batch_size, 'TEST0')

Epoch 0
Epoch loss : 5900851.0000 Accuracy : 0.5042 Test loss :1665908.2500 Accuracy test : 0.5067
TEST0
Epoch 1
Epoch loss : 5707247.5000 Accuracy : 0.5205 Test loss :1585508.0000 Accuracy test : 0.5305
Epoch 2
Epoch loss : 5413921.0000 Accuracy : 0.5453 Test loss :1482607.7500 Accuracy test : 0.5610
Epoch 3
Epoch loss : 5058105.0000 Accuracy : 0.5752 Test loss :1365307.3750 Accuracy test : 0.5957
Epoch 4
Epoch loss : 4655313.5000 Accuracy : 0.6090 Test loss :1245807.1250 Accuracy test : 0.6311
Epoch 5
Epoch loss : 4253524.0000 Accuracy : 0.6426 Test loss :1119906.7500 Accuracy test : 0.6684
Epoch 6
Epoch loss : 3852691.7500 Accuracy : 0.6765 Test loss :986206.4375 Accuracy test : 0.7080
Epoch 7
Epoch loss : 3454186.7500 Accuracy : 0.7100 Test loss :872806.1250 Accuracy test : 0.7416
Epoch 8
Epoch loss : 3099370.5000 Accuracy : 0.7397 Test loss :773605.7500 Accuracy test : 0.7709
Epoch 9
Epoch loss : 2788279.0000 Accuracy : 0.7656 Test loss :698305.5000 Accuracy test : 0.7932
Epoch 10

()